In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications import DenseNet201
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import gc

DATASET_DIR = '/content/drive/MyDrive/brain_tumor_project /brain/split_dataset'
BATCH_SIZE = 16
IMAGE_SIZE = 224
INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)
EPOCHS = 50
LEARNING_RATE = 1e-4


PROJECTION_DIM = 256
NUM_TRANSFORMER_LAYERS = 4
NUM_HEADS = 8
FF_DIM = PROJECTION_DIM * 4

print("--- Setting up Data Generators ---")
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
    shear_range=0.1, zoom_range=0.1, horizontal_flip=True, fill_mode='nearest'
)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    DATASET_DIR + "/training", target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=True)
val_generator = val_datagen.flow_from_directory(
    DATASET_DIR + "/val", target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)
test_generator = test_datagen.flow_from_directory(
    DATASET_DIR + "/test", target_size=(IMAGE_SIZE, IMAGE_SIZE),
    batch_size=BATCH_SIZE, class_mode="categorical", shuffle=False)

NUM_CLASSES = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print(f"Found {NUM_CLASSES} classes: {class_names}")



def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    x = layers.LayerNormalization(epsilon=1e-6)(inputs)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.Dropout(dropout)(x)
    res = x + inputs
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="gelu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    return x + res

def build_msca_denseformer(input_shape, num_classes):
    print("--- Building The MSCA-DenseFormer ---")

    base_model = DenseNet201(include_top=False, weights='imagenet', input_shape=input_shape)
    mid_features_layer = base_model.get_layer('conv4_block24_concat').output
    final_features_layer = base_model.output
    feature_extractor = models.Model(inputs=base_model.input, outputs=[mid_features_layer, final_features_layer])


    feature_extractor.trainable = True
    for layer in feature_extractor.layers[:427]: # Freeze up to conv5_block1
        layer.trainable = False

    inputs = layers.Input(shape=input_shape)

    mid_features, final_features = feature_extractor(inputs)

    mid_seq = layers.Reshape((-1, mid_features.shape[-1]))(mid_features)
    final_seq = layers.Reshape((-1, final_features.shape[-1]))(final_features)

    mid_proj = layers.Dense(PROJECTION_DIM, name="mid_projection")(mid_seq)
    final_proj = layers.Dense(PROJECTION_DIM, name="final_projection")(final_seq)


    cross_attention_output = layers.MultiHeadAttention(
        num_heads=NUM_HEADS, key_dim=PROJECTION_DIM // NUM_HEADS, name="cross_attention"
    )(query=final_proj, value=mid_proj, key=mid_proj)
    fused_seq = layers.Add(name="fusion")([final_proj, cross_attention_output])
    x = fused_seq
    seq_len = x.shape[1]
    pos_embed = layers.Embedding(input_dim=seq_len, output_dim=PROJECTION_DIM)(tf.range(seq_len))
    x = x + pos_embed


    for i in range(NUM_TRANSFORMER_LAYERS):
        x = transformer_encoder(x, PROJECTION_DIM // NUM_HEADS, NUM_HEADS, FF_DIM)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    return model



msca_denseformer = build_msca_denseformer(INPUT_SHAPE, NUM_CLASSES)

msca_denseformer.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=1e-5),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

msca_denseformer.summary()

checkpoint = ModelCheckpoint("msca_denseformer_best.keras", save_best_only=True, monitor="val_accuracy", mode="max", verbose=1)
early_stop = EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1)

history = msca_denseformer.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop, reduce_lr]
)


print("\n--- Final Evaluation on Test Data ---")
results = msca_denseformer.evaluate(test_generator, verbose=1)


print("\n--- Final Evaluation on Val Data ---")
results = msca_denseformer.evaluate(val_generator, verbose=1)


print("\n--- Generating Classification Report ---")
test_preds_proba = msca_denseformer.predict(test_generator)
y_pred = np.argmax(test_preds_proba, axis=1)
y_true = test_generator.classes
test_accuracy = accuracy_score(y_true, y_pred)

print(f"\nFinal Accuracy on Test Data: {test_accuracy*100:.2f}%")
print("\nFull Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

del msca_denseformer
gc.collect()

--- Setting up Data Generators ---
Found 7035 images belonging to 4 classes.
Found 875 images belonging to 4 classes.
Found 882 images belonging to 4 classes.
Found 4 classes: ['glioma', 'meningioma', 'no_tumor', 'pituitary']
--- Building The MSCA-DenseFormer ---
74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional          │ [(None, 14, 14,   │ 18,321,984 │ input_layer_1[0]… │
│ (Functional)        │ 1024), (None, 7,  │            │                   │
│                     │ 7, 1920)]         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 49, 1920)  │          0 │ functional[0][1]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 196, 1024) │          0 │ functional[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_projection    │ (None, 49, 256)   │    491,776 │ reshape_1[0][0]   │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mid_projection      │ (None, 196, 256)  │    262,400 │ reshape[0][0]     │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cross_attention     │ (None, 49, 256)   │    263,168 │ mid_projection[0… │
│ (MultiHeadAttentio… │                   │            │ final_projection… │
│                     │                   │            │ mid_projection[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fusion (Add)        │ (None, 49, 256)   │          0 │ final_projection… │
│                     │                   │            │ cross_attention[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 49, 256)   │          0 │ fusion[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 49, 256)   │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 49, 256)   │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 49, 256)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 49, 256)   │          0 │ dropout_2[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 49, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 49, 1024)  │    263,168 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 49, 1024)  │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 49, 256)   │    262,400 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 49, 256)   │          0 │ conv1d_1[0][0],   │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 22,531,780 (85.95 MB)

 Trainable params: 14,575,492 (55.60 MB)

 Non-trainable params: 7,956,288 (30.35 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
440/440 ━━━━━━━━━━━━━━━━━━━━ 0s 5s/step - accuracy: 0.6516 - loss: 1.0982
Epoch 1: val_accuracy improved from -inf to 0.85600, saving model to msca_denseformer_best.keras
440/440 ━━━━━━━━━━━━━━━━━━━━ 2760s 6s/step - accuracy: 0.6518 - loss: 1.0978 - val_accuracy: 0.8560 - val_loss: 0.6626 - learning_rate: 1.0000e-04
Epoch 2/50
440/440 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.8610 - loss: 0.6816
Epoch 2: val_accuracy improved from 0.85600 to 0.88000, saving model to msca_denseformer_best.keras
440/440 ━━━━━━━━━━━━━━━━━━━━ 130s 296ms/step - accuracy: 0.8610 - loss: 0.6816 - val_accuracy: 0.8800 - val_loss: 0.6061 - learning_rate: 1.0000e-04
Epoch 3/50
440/440 ━━━━━━━━━━━━━━━━━━━━ 0s 276ms/step - accuracy: 0.8945 - loss: 0.6157
Epoch 3: val_accuracy improved from 0.88000 to 0.89371, saving model to msca_denseformer_best.keras
440/440 ━━━━━━━━━━━━━━━━━━━━ 131s 297ms/step - accuracy: 0.8945 - loss: 0.6156 - val_accuracy: 0.8937 - val_loss: 0.5636 - learning_rate: 1.0000e-

2163

[Errno 2] No such file or directory: '/content/drive/MyDrive/brain_tumor_project'
/content


In [ ]:
%cd /content/drive/MyDrive/brain_tumor_project/brain/

[Errno 2] No such file or directory: '/content/drive/MyDrive/brain_tumor_project/brain/'
/content


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.metrics import Precision, Recall
import numpy as np
from sklearn.metrics import classification_report, accuracy_score


DATASET_DIR = '/content/drive/MyDrive/brain_tumor_project /brain/split_dataset'
BATCH_SIZE = 16


IMAGE_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2

PROJECTION_DIM = 176
TRANSFORMER_LAYERS = 3
NUM_HEADS = 4

MLP_HEAD_UNITS = [
    PROJECTION_DIM * 2,
    PROJECTION_DIM,
]

MLP_DROPOUT = 0.1
ATTENTION_DROPOUT = 0.1
EPOCHS = 25


train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory( DATASET_DIR + "/training", target_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE,
    class_mode="categorical",shuffle=False
)
val_generator = val_datagen.flow_from_directory(DATASET_DIR + "/val",
    target_size=(IMAGE_SIZE, IMAGE_SIZE), batch_size=BATCH_SIZE, class_mode="categorical",shuffle=False
)
test_generator = test_datagen.flow_from_directory(DATASET_DIR + "/test", target_size=(IMAGE_SIZE, IMAGE_SIZE),
 batch_size=BATCH_SIZE,class_mode="categorical", shuffle=False
)

NUM_CLASSES = train_generator.num_classes
INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)
def mlp(x, hidden_units, dropout_rate):
    for units in hidden_units:
        x = layers.Dense(units, activation=tf.nn.gelu)(x)
        x = layers.Dropout(dropout_rate)(x)
    return x


@tf.keras.utils.register_keras_serializable()
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        patches = tf.reshape(patches, [batch_size, -1, patch_dims])
        return patches

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size})
        return config


@tf.keras.utils.register_keras_serializable()
class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim  # Store for get_config
        self.projection = layers.Dense(units=projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches, "projection_dim": self.projection_dim})
        return config



def build_vit_classifier(
    input_shape, patch_size, num_patches, projection_dim,
    transformer_layers, num_heads, mlp_head_units, num_classes,
):
    inputs = layers.Input(shape=input_shape)
    patches = Patches(patch_size)(inputs)
    encoded_patches = PatchEncoder(num_patches, projection_dim)(patches)
    x = encoded_patches

    for _ in range(transformer_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads,key_dim=projection_dim//num_heads ,dropout=ATTENTION_DROPOUT
        )(x1, x1)
        x2 = layers.Add()([attention_output, x])
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
        x3 = mlp(x3, hidden_units=mlp_head_units, dropout_rate=MLP_DROPOUT)
        x = layers.Add()([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6)(x)
    representation = layers.GlobalAveragePooling1D()(representation)
    representation = layers.Dropout(0.5)(representation)
    outputs = layers.Dense(num_classes, activation="softmax")(representation)
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

vit_classifier = build_vit_classifier(
    input_shape=INPUT_SHAPE, patch_size=PATCH_SIZE, num_patches=NUM_PATCHES,
    projection_dim=PROJECTION_DIM, transformer_layers=TRANSFORMER_LAYERS,
    num_heads=NUM_HEADS, mlp_head_units=MLP_HEAD_UNITS, num_classes=NUM_CLASSES,
)
vit_classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
    loss="categorical_crossentropy",
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)
vit_classifier.summary()
checkpoint = ModelCheckpoint(
    "vit_brain_tumor_best_1M.keras",
    save_best_only=True, monitor="val_accuracy", mode="max"
)
early_stop = EarlyStopping(
    monitor="val_accuracy", patience=15, restore_best_weights=True
)

history = vit_classifier.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)





best_model = models.load_model("vit_brain_tumor_best_1M.keras") # This will now work correctly
results = best_model.evaluate(test_generator, verbose=1)
for name, value in zip(best_model.metrics_names, results):
    print(f"{name}: {value:.4f}")

test_preds_proba = best_model.predict(test_generator)
y_pred = np.argmax(test_preds_proba, axis=1)
y_true = test_generator.classes
test_accuracy = accuracy_score(y_true, y_pred)
print(f"\n Accuracy on Test Data: {test_accuracy*100:.2f}%")
class_names = list(train_generator.class_indices.keys())
print("\nFull Classification Report :")
print(classification_report(y_true, y_pred, target_names=class_names))

Found 7035 images belonging to 4 classes.
Found 875 images belonging to 4 classes.
Found 882 images belonging to 4 classes.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patches (Patches)   │ (None, None, 768) │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_encoder       │ (None, 196, 176)  │    169,840 │ patches[0][0]     │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 196, 176)  │        352 │ patch_encoder[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 176)  │    124,608 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 196, 176)  │          0 │ multi_head_atten… │
│                     │                   │            │ patch_encoder[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 176)  │        352 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 196, 352)  │     62,304 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 196, 352)  │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 196, 176)  │     62,128 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 196, 176)  │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 196, 176)  │          0 │ dropout_2[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 176)  │        352 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 176)  │    124,608 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 196, 176)  │          0 │ multi_head_atten… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 176)  │        352 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 196, 352)  │     62,304 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 196, 352)  │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 196, 176)  │     62,128 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 920,132 (3.51 MB)

 Trainable params: 920,132 (3.51 MB)

 Non-trainable params: 0 (0.00 B)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/25
 35/440 ━━━━━━━━━━━━━━━━━━━━ 23:55 4s/step - accuracy: 0.1709 - loss: 2.9351 - precision: 0.1615 - recall: 0.1317